In [4]:
from netCDF4 import Dataset
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import datetime
import pickle
from scipy.interpolate import griddata
from mpl_toolkits.basemap import Basemap
import os
from sklearn.linear_model import LinearRegression
from matplotlib.ticker import FixedLocator, MultipleLocator, IndexLocator
from pyproj import CRS, Transformer

In [2]:
lon_min_era5, lon_max_era5, lat_min_era5, lat_max_era5 = 35, 105, 66, 82
lat1, lat2 = (90-lat_max_era5)*4, (90-lat_min_era5)*4+1
lon1, lon2 = lon_min_era5*4, lon_max_era5*4+1

In [3]:
month = '2024-09'
file = f'/mnt/hippocamp/DATA/ERA5/w10/era5_uv10m_{month}.nc'
data = Dataset(file, 'r')

tt = np.asarray(data.variables['valid_time'])
time = np.asarray([datetime.datetime(1970, 1, 1, 0, 0, 0) + datetime.timedelta(seconds=int(t)) for t in tt])
u10 = data.variables['u10'][:,lat1:lat2,lon1:lon2]
v10 = data.variables['v10'][:,lat1:lat2,lon1:lon2]

longitude = np.array(data.variables['longitude'][lon1:lon2])
latitude = np.array(data.variables['latitude'][lat1:lat2])
lon_grid, lat_grid = np.meshgrid(longitude, latitude)

data.close()

In [ ]:
def apply_offset_aeqd(lat0, lon0, dx_km, dy_km):
    """
    lat0, lon0: исходная широта/долгота в градусах
    dx_km: смещение на восток (км), положительное вправо
    dy_km: смещение на север (км), положительное вверх
    Возвращает: (lat2, lon2) — новые координаты в градусах
    """
    
    # Геодезическая (широта/долгота) и локальная метрика (метры)
    crs_geodetic = CRS.from_epsg(4326)
    crs_local = CRS.from_proj4(f"+proj=aeqd +lat_0={lat0} +lon_0={lon0} +datum=WGS84 +units=m +no_defs")

    to_local = Transformer.from_crs(crs_geodetic, crs_local, always_xy=True)
    to_geo = Transformer.from_crs(crs_local, crs_geodetic, always_xy=True)

    # Исходная точка в локальных координатах (метры)
    x0, y0 = to_local.transform(lon0, lat0)

    # Применяем смещение (переводим км в метры)
    x1 = x0 + dx_km * 1000.0
    y1 = y0 + dy_km * 1000.0

    # Обратно в широту/долготу
    lon2, lat2 = to_geo.transform(x1, y1)

    # Нормализуем долготу в диапазон [-180, 180] при необходимости
    # lon2 = (lon2 + 180.0) % 360.0 - 180.0

    return lat2, lon2